##### date_sub()

- is used to **subtract** a specified **number of days** from a **date** column.

**Syntax**

     date_sub(start_date, days)

- **Subtracts** a specified **number of days** to a **date** column.
  - `Positive value` → `Past date`
  - `Negative value` → `Future date`

|    Argument    |      Description                       |
|----------------|----------------------------------------|
| **start_date** | `Column containing date`               |
| **days**       | `Number of days to subtract (integer)` |

##### 1) Subtract Days from Current Date

In [0]:
from pyspark.sql.functions import col, lit, current_date, date_sub, datediff

In [0]:
df_sub = (spark.range(1)
  .withColumn("current_date", current_date())
  # Subtracting Days from a Date
  .withColumn("10_days_before", date_sub(current_date(), 10))
  .withColumn('diff_days_01', datediff(col('current_date'), col('10_days_before')))
  # Adding Days to a Date 
  .withColumn("10_days_after", date_sub(current_date(), -10))
  .withColumn('diff_days_02', datediff(col('current_date'), col('10_days_after')))
  # Calculating the date 90 days before
  .withColumn("90_days_before", date_sub(lit('2026-03-07'), 90))
  .withColumn('diff_days_03', datediff(col('current_date'), col('90_days_before')))
  .withColumn("1_day_after", date_sub(current_date(), lit(-1)))
  .withColumn('diff_days_04', datediff(col('current_date'), col('1_day_after'))))

display(df_sub)
df_sub.printSchema()

id,current_date,10_days_before,diff_days_01,10_days_after,diff_days_02,90_days_before,diff_days_03,1_day_after,diff_days_04
0,2026-03-07,2026-02-25,10,2026-03-17,-10,2025-12-07,90,2026-03-08,-1


root
 |-- id: long (nullable = false)
 |-- current_date: date (nullable = false)
 |-- 10_days_before: date (nullable = false)
 |-- diff_days_01: integer (nullable = false)
 |-- 10_days_after: date (nullable = false)
 |-- diff_days_02: integer (nullable = false)
 |-- 90_days_before: date (nullable = true)
 |-- diff_days_03: integer (nullable = true)
 |-- 1_day_after: date (nullable = false)
 |-- diff_days_04: integer (nullable = false)



##### 2) Subtract Days from Existing Date Column

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

data = [("2026-02-16",), ("2026-02-10",), ("2026-02-05",), ("2026-01-05",), ("2026-01-10",), ("2026-01-15",),]

df_sub_date = spark.createDataFrame(data, ['order_date'])

df_sub_date_01 = df_sub_date \
    .withColumn("order_date", col("order_date").cast("date")) \
    .withColumn("7_days_before", date_sub(col("order_date"), 7)) \
    .withColumn("diff_days_01", datediff(col("order_date"), col("7_days_before"))) \
    .withColumn("30_days_before", date_sub(current_date(), 30)) \
    .withColumn("diff_days_02", datediff(current_date(), col("30_days_before")))

display(df_sub_date_01)

order_date,7_days_before,diff_days_01,30_days_before,diff_days_02
2026-02-16,2026-02-09,7,2026-02-05,30
2026-02-10,2026-02-03,7,2026-02-05,30
2026-02-05,2026-01-29,7,2026-02-05,30
2026-01-05,2025-12-29,7,2026-02-05,30
2026-01-10,2026-01-03,7,2026-02-05,30
2026-01-15,2026-01-08,7,2026-02-05,30


##### 3) Using date_sub() in Filter Condition
- `Get records from the last 30 days`
- This `filters` rows where `order_date` is within the last `30 days`.

In [0]:
df_filtered = df_sub_date \
    .filter(col("order_date") >= date_sub(current_date(), 30))
display(df_filtered)

order_date
2026-02-16
2026-02-10
2026-02-05


##### 4) Dynamic Subtraction Using Column Value

In [0]:
data = [("2026-02-16", 5), ("2026-02-16", 15), ("2026-02-16", 30), ("2026-02-16", 60), ("2026-02-16", 90)]

from pyspark.sql.types import StructField, StructType, StringType, IntegerType

schema = StructType([
    StructField("order_date", StringType(), True),
    StructField("days_to_subtract", IntegerType(), True)
])

df_dynamic = spark.createDataFrame(data, schema) \
    .withColumn("order_date", col("order_date").cast("date")) \
    .withColumn("new_date", date_sub(col("order_date"), col("days_to_subtract")))

display(df_dynamic)

order_date,days_to_subtract,new_date
2026-02-16,5,2026-02-11
2026-02-16,15,2026-02-01
2026-02-16,30,2026-01-17
2026-02-16,60,2025-12-18
2026-02-16,90,2025-11-18
